In [0]:
WITH price_wide AS (
    SELECT * FROM VALUES
        --  category, Oct-25 price, Nov-26 price
        ('1',   15.75, 16.00),
        ('2',   15.25, 15.50),
        ('3',   14.75, 15.00),
        ('4',   13.75, 14.00),
        ('5',   13.75, 14.00),
        ('6',   12.75, 13.00),
        ('8',   15.75, 16.00),
        ('9',   14.25, 14.50),
        ('11',  12.25, 12.50),
        ('12',  12.75, 13.00),
        ('14',  14.00, 14.25),
        ('15',  13.25, 13.50),
        ('16',  12.50, 12.75),
        ('17',  12.75, 13.00),
        ('18A', 13.75, 14.00),
        ('19',  11.25, 11.50),
        ('21',  15.25, 15.50),
        ('21J', 16.00, 16.25),
        ('22',  14.50, 14.75),
        ('23',  13.50, 13.75),
        ('24',  12.50, 12.75),
        ('24A', 14.50, 14.75),
        ('25',  12.25, 12.50),
        ('26',  13.75, 14.00),
        ('26A', 14.25, 14.50),
        ('27',  13.25, 13.50),
        ('28',  12.25, 12.50),
        ('29',  11.25, 11.50),
        ('30',  12.25, 12.50),
        ('31',  13.25, 13.50),
        ('32',  12.75, 13.00),
        ('33',  12.25, 12.50),
        ('34',  11.25, 11.50)
    AS t(Price_Category, price_oct25, price_nov26)
),

price_category_dict AS (
    SELECT Price_Category, DATE'2025-10-01' AS Start_Date, DATE'2026-10-31' AS End_Date,
           CAST(price_oct25 AS DECIMAL(18,2)) AS Adult_General
    FROM price_wide
    UNION ALL
    SELECT Price_Category, DATE'2026-11-01' AS Start_Date, DATE'9999-12-31' AS End_Date,   -- "Current"
           CAST(price_nov26 AS DECIMAL(18,2)) AS Adult_General
    FROM price_wide
),

-- Price per location via pricing tier
price_hist AS (
    SELECT l.locationid AS LocationId, d.Adult_General, d.Start_Date, d.End_Date
    FROM data_exploration_hub.marketing_hub.mkt_pricing_tier_by_loc l
    JOIN price_category_dict d
      ON UPPER(TRIM(l.pricing_tier)) = d.Price_Category
),

-- Members active in EVERY snapshot from Jan 1, 2026 to yesterday
snap_ytd AS (
    SELECT CAST(cde_id AS INT) AS CDE_ID, snapshot_date_id, renewal_status
    FROM cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot
    WHERE snapshot_date_id BETWEEN 20260101
                               AND CAST(date_format(date_sub(current_date(), 1), 'yyyyMMdd') AS INT)
),

active_cc AS (
    SELECT CDE_ID
    FROM snap_ytd
    GROUP BY CDE_ID
    HAVING COUNT(DISTINCT CASE WHEN renewal_status LIKE 'Active%' THEN snapshot_date_id END)
         = (SELECT COUNT(DISTINCT snapshot_date_id) FROM snap_ytd)
),

txn AS (
    SELECT
        t.CDE_ID, t.VisitDateId, t.LocationId, t.Net_Revenue, t.Quantity,
        t.TicketFLAG, t.ConcessionFLAG, t.COTFLAG, t.CineClub_Discount_Type_ID,
        to_date(CAST(t.VisitDateId AS STRING), 'yyyyMMdd') AS visit_date
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    JOIN active_cc a ON t.CDE_ID = a.CDE_ID
    WHERE t.VisitDateId BETWEEN 20260101 AND CAST(date_format(current_date(), 'yyyyMMdd') AS INT)
      AND t.CineClubFLAG = 'Yes'
),

-- One price per (location, date) — guards against duplicate tier rows
price_lookup AS (
    SELECT d.LocationId, d.visit_date, p.Adult_General
    FROM (SELECT DISTINCT LocationId, visit_date FROM txn) d
    JOIN price_hist p
      ON  d.LocationId = p.LocationId
      AND d.visit_date BETWEEN p.Start_Date AND p.End_Date
    QUALIFY ROW_NUMBER() OVER (PARTITION BY d.LocationId, d.visit_date ORDER BY p.Start_Date DESC) = 1
),

member_savings AS (
    SELECT
        t.CDE_ID,
        ROUND(SUM(CASE WHEN t.CineClub_Discount_Type_ID IN (5, 6)
                       THEN t.Quantity * (COALESCE(pl.Adult_General, 9.99) - 9.99) ELSE 0 END), 2) AS ticket_savings_999,
        ROUND(SUM(CASE WHEN t.CineClub_Discount_Type_ID IN (7, 8)
                       THEN t.Quantity * (COALESCE(pl.Adult_General, 10.99) - 10.99) ELSE 0 END), 2) AS ticket_savings_1099,
        ROUND(SUM(CASE WHEN t.ConcessionFLAG = 'Yes' AND t.CineClub_Discount_Type_ID = 2
                       THEN t.Net_Revenue * 0.25 ELSE 0 END), 2) AS concession_savings,
        ROUND(SUM(CASE WHEN t.ConcessionFLAG = 'Yes' AND t.CineClub_Discount_Type_ID = 3
                       THEN t.Net_Revenue * 0.25 ELSE 0 END), 2) AS gaming_savings,
        COUNT(DISTINCT CASE WHEN t.TicketFLAG = 'Yes' AND t.COTFLAG = 'Yes'
                            THEN concat_ws('|', CAST(t.VisitDateId AS STRING), CAST(t.LocationId AS STRING)) END) AS online_booking_savings
    FROM txn t
    LEFT JOIN price_lookup pl
      ON t.LocationId = pl.LocationId AND t.visit_date = pl.visit_date
    GROUP BY t.CDE_ID
),

member_total AS (
    SELECT a.CDE_ID,
           m.CDE_ID IS NOT NULL                       AS had_activity,
           COALESCE(m.ticket_savings_999, 0)          AS ticket_savings_999,
           COALESCE(m.ticket_savings_1099, 0)         AS ticket_savings_1099,
           COALESCE(m.concession_savings, 0)          AS concession_savings,
           COALESCE(m.gaming_savings, 0)              AS gaming_savings,
           COALESCE(m.online_booking_savings, 0)      AS online_booking_savings,
           GREATEST(ROUND(COALESCE(m.ticket_savings_999, 0) + COALESCE(m.ticket_savings_1099, 0)
                          + COALESCE(m.concession_savings, 0) + COALESCE(m.gaming_savings, 0)
                          + COALESCE(m.online_booking_savings, 0), 2), 0) AS total_savings
    FROM active_cc a
    LEFT JOIN member_savings m ON a.CDE_ID = m.CDE_ID
)

SELECT
    COUNT(*)                                                         AS full_year_members,
    SUM(CASE WHEN had_activity THEN 1 ELSE 0 END)                    AS members_with_cc_activity,
    ROUND(AVG(ticket_savings_999), 2)                                AS avg_ticket_savings_999,
    ROUND(AVG(ticket_savings_1099), 2)                               AS avg_ticket_savings_1099,
    ROUND(AVG(concession_savings), 2)                                AS avg_concession_savings,
    ROUND(AVG(gaming_savings), 2)                                    AS avg_gaming_savings,
    ROUND(AVG(online_booking_savings), 2)                            AS avg_online_booking_savings,
    ROUND(AVG(total_savings), 2)                                     AS avg_total_savings_all,
    ROUND(AVG(CASE WHEN had_activity THEN total_savings END), 2)     AS avg_total_savings_users_only,
    ROUND(percentile_approx(total_savings, 0.5), 2)                  AS median_total_savings
FROM member_total;